In [1]:

# ---------------------------------------------------------------------------
# Setup: s3fs + small-B grid (24 x 33 x 24; x,z periodic uniform, y clustered)
#   Prediction datasets:  simulations/couette_ds/<name>/
# ---------------------------------------------------------------------------
import numpy as np
import zarr, s3fs, h5py
from scipy.linalg import solve_banded

S3_BUCKET = "simulations"
PRED_ROOT = f"{S3_BUCKET}/couette_ds"          # where prediction datasets live
fs = s3fs.S3FileSystem(profile='default', client_kwargs={'endpoint_url': 'http://localhost:9000'})

H5 = "/data1/dataToYouri/couette_dns_ml_f32.h5"
with h5py.File(H5, 'r') as f:
    x_grid, y_grid, z_grid = f['x'][:], f['y'][:], f['z'][:]
Nx, Ny, Nz = len(x_grid), len(y_grid), len(z_grid)
Lx = x_grid[-1] + (x_grid[1]-x_grid[0])      # periodic: last + one cell
Ly = y_grid[-1] - y_grid[0]
Lz = z_grid[-1] + (z_grid[1]-z_grid[0])
print(f"grid Nx={Nx} Lx={Lx} Ly={Ly:.4f} Ny={Ny} Nz={Nz} Lz={Lz:.4f} (x,z periodic, y clustered)")


grid  Nx=24 Ly=2.0000 Ny=33 Lz=3.7699 (x,z periodic, y clustered)


In [7]:

# ---------------------------------------------------------------------------
# High-accuracy gradient (port of check_export.get_grad_func, re-derived for small-B)
#   x,z : FFT derivative (periodic, uniform)
#   y   : 6th-order compact finite difference (uniform dy) * exact metric factor
#
# small-B y-grid is EXACTLY  y = -cos(pi*eta), eta = i/(Ny-1)   (verified to 1e-16)
#   => dy/deta = pi/(Ny-1) * sin(pi*eta)   (exact, no hyperbolic-tangent fit)
#   (check_export's tanh formula is a fit for the 128^3 grid, not exact here.)
# ---------------------------------------------------------------------------

def broadcast_to(source, target, index):
    target_shape = [1] * target.ndim
    target_shape[index] = source.shape[0]
    return source.reshape(target_shape)

def derive_fft(u, direction, h):
    n = u.shape[direction]
    k = 2.0 * np.pi * np.fft.fftfreq(n, d=h)
    k_b = broadcast_to(k, u, direction)
    uh = np.fft.fftn(u, axes=(direction,))
    return np.fft.ifftn(1.0j * k_b * uh, axes=(direction,)).real

def derive_compact_6th(field, direction, dy):
    """6th-order compact derivative along `direction` (uniform spacing dy)."""
    x = np.moveaxis(field, direction, 0)
    shape = x.shape
    rhs = np.zeros_like(x)
    length = x.shape[0]
    ab = np.zeros((3, length))
    a, b, c = -1/36, -7/9, 1/3
    rhs[2:-2] = a*(x[0:-4] - x[4:]) + b*(x[1:-3] - x[3:-1])
    rhs[0]   = (-5/2)*x[0] + 2*x[1] + (1/2)*x[2]
    rhs[1]   = (3/4)*(x[2] - x[0])
    rhs[-2]  = -(3/4)*(x[-3] - x[-1])
    rhs[-1]  = -(-5/2)*x[-1] - 2*x[-2] - (1/2)*x[-3]
    rhs[:] = rhs / dy
    ab[0, 2:] = c;  ab[1, :] = 1;  ab[2, :-2] = c
    ab[0, 1] = 2;  ab[0, 2] = 1/4;  ab[0, -1] = 1/4
    ab[2, -2] = 2; ab[2, 0] = 1/4;  ab[2, -3] = 1/4
    out = solve_banded((1, 1), ab, rhs.reshape(length, -1))
    return np.moveaxis(out.reshape(shape), 0, direction)

def metric_factor_y(y, Ly):
    """Exact dy/deta for  y = -cos(pi*eta), eta = i/(Ny-1):  dy/deta = pi/(Ny-1)*sin(pi*eta)."""
    ny = len(y); nym = ny - 1
    eta = np.arange(ny)/nym
    return (np.pi/nym)*np.sin(np.pi*eta)

def make_grad(Lx, Ly, Lz, Nx, Ny, Nz, ppy):
    dx = Lx / Nx; dz = Lz / Nz
    def grad(field):
        T, C, X, Y, Z = field.shape
        g = np.zeros((T, C, 3, X, Y, Z), dtype=field.dtype)
        g[:, :, 0] = derive_fft(field, 2, dx)                                   # d/dx
        g[:, :, 1] = derive_compact_6th(field, 3, Ly/(Ny-1)) * ppy[None,None,None,:,None]  # d/dy
        g[:, :, 2] = derive_fft(field, 4, dz)                                   # d/dz
        return g
    return grad

ppy = metric_factor_y(y_grid, Ly)
grad = make_grad(Lx, Ly, Lz, Nx, Ny, Nz, ppy)
print("grad operator ready;  ppy range", ppy.min().round(4), ppy.max().round(4))


grad operator ready;  ppy range 0.0 0.0982


In [40]:

# ---------------------------------------------------------------------------
# Physical check suite  (each takes a field [T,3,X,Y,Z] + optional reference)
#   All computed in PHYSICAL space (fields must be unnormalized).
# ---------------------------------------------------------------------------

def rel_rmse(X, Y):
    return np.sqrt(np.mean((X - Y) ** 2)) / np.sqrt(np.mean(Y ** 2))

def check_divergence(field):
    """Incompressibility: div u = du_x/dx + du_y/dy + du_z/dz  (should be ~0).
    NOTE: source DNS at Re_tau~200 has an intrinsic floor ~0.2 (see source benchmark)
    — the meaningful comparison for a PREDICTION is against its own source, not 0."""
    g = grad(field)
    div = g[:, 0, 0] + g[:, 1, 1] + g[:, 2, 2]
    return {"mean|div|": float(np.abs(div).mean()),
            "mean(div)": float(div.mean()),
            "max|div|":  float(np.abs(div).max())}

def check_wall_bc(field):
    """No-slip at both walls on the DEVIATION field: u|_wall = 0 (all 3 components).
    field is [T, C, X, Y, Z]; wall slices are field[:, :, :, 0, :] and [:, :, :, -1, :]."""
    wall_a = field[:, :, :, 0, :]    # y=-1  [T,C,X,Z]
    wall_b = field[:, :, :, -1, :]   # y=+1  [T,C,X,Z]
    return {"wall_a |u| mean": float(np.abs(wall_a).mean()),
            "wall_b |u| mean": float(np.abs(wall_b).mean()),
            "wall_a u_y mean": float(np.abs(wall_a[:, 1]).mean()),
            "wall_b u_y mean": float(np.abs(wall_b[:, 1]).mean())}

def check_Q(field, Q_ref=None):
    """Q-criterion = -0.5 * tr((grad u)^2). Compare to reference if given."""
    g = grad(field)
    Q = -0.5 * np.einsum('tij...,tji...->t...', g[:, :3], g[:, :3])
    out = {"Q mean": float(Q.mean()), "Q rms": float(np.sqrt((Q**2).mean()))}
    if Q_ref is not None:
        out["rel_rmse(Q, ref)"] = float(rel_rmse(Q, Q_ref))
    return out

def check_re_tau(field, nu, tau_mean_a=None, tau_mean_b=None):
    """Re_tau = u_tau/nu,  u_tau = sqrt(nu * |tau_mean|).
    Uses STORED wall/tau_mean (Chebyshev-exact, total shear = laminar + turbulent)
    when provided; falls back to FD on the deviation field otherwise."""
    if tau_mean_a is not None and tau_mean_b is not None:
        tau = np.abs(np.mean([tau_mean_a, tau_mean_b], axis=0))
        u_tau = float(np.sqrt(nu * np.mean(tau)))
    else:
        y_b = y_grid[None, None, None, :, None]
        u_tot = field[:, 0] + y_b
        d0 = y_grid[1] - y_grid[0]; d1 = y_grid[2] - y_grid[1]; r0 = d0/(d0+d1)
        shear_a = (-1/r0 * u_tot[:, :, 0] + (1 + 1/r0) * u_tot[:, :, 1]) / d0
        dN = y_grid[-1] - y_grid[-2]; dNm = y_grid[-2] - y_grid[-3]; rN = dN/(dN+dNm)
        shear_b = (u_tot[:, :, -1] * (1 + 1/rN) - u_tot[:, :, -2] * (1/rN)) / dN
        tau = nu * np.abs(np.mean([shear_a, shear_b], axis=0))
        u_tau = float(np.sqrt(np.mean(tau)))
    return {"u_tau": u_tau, "Re_tau": u_tau / nu}

def check_stationarity(field, t_step=1):
    """Stationarity: d<field>/dt should be ~0 for a statistically stationary DNS."""
    f = field.mean(0)   # [C,X,Y,Z]
    d = np.abs(f[t_step:] - f[:-t_step]).mean()
    rms = np.sqrt((f**2).mean())
    return {"d<field>/dt rms": float(d), "rel. d<field>/dt": float(d / rms)}

def run_all_checks(field, nu, Q_ref=None, tau_mean_a=None, tau_mean_b=None):
    """Run the full suite on a physical-space field [T,3,X,Y,Z] (deviation).
    tau_mean_a/b: stored wall/tau_mean [T] for exact Re_tau (source only)."""
    return {"divergence":   check_divergence(field),
            "wall_bc":      check_wall_bc(field),
            "Q":            check_Q(field, Q_ref),
            "re_tau":       check_re_tau(field, nu, tau_mean_a, tau_mean_b),
            "stationarity": check_stationarity(field)}


In [33]:

# ---------------------------------------------------------------------------
# CELL: source data (small-B, unnormalized) — the per-sample reference list.
#   src[i]  <->  realization REALS[i//5],  sample (i%5)*300
#   Used both for the floor benchmark (cell 4) and per-sample pred-vs-source.
# ---------------------------------------------------------------------------
import time, trackio

NU = 1/400           # viscosity: Re = 400 (dataset attr),  Re_tau ~ 34 (stored wall/tau_mean)
REALS = ['R13','R14','R15','R18','R19','R21','R22','R23','R26','R28','R3','R4','R5']
KEYS = 'u', 'v', 'w', 'p_total'
T_SUB = slice(None, None, 300)   # every 300th sample, as in the train notebooks

def extract(real):
    with h5py.File(H5, 'r') as f:
        return np.stack([np.array(f[real][k]) for k in KEYS], axis=1)   # [T,4,X,Y,Z]

SRC = np.concatenate([extract(r)[T_SUB][:, :3] for r in REALS], axis=0)   # [T,3,X,Y,Z]
print("source list:", SRC.shape, SRC.dtype, f"({len(REALS)} realizations x 5 samples)")

def flatten(d, prefix=""):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flatten(v, f"{prefix}{k}."))
        else:
            out[f"{prefix}{k}"] = v
    return out

def log_checks(d, tag, step, extra_cfg=None):
    trackio.log({**flatten(d, f"{tag}_"), "step": step}, step=step)

print("ready.")


source list: (61, 3, 24, 33, 24) float32 (13 realizations x 5 samples)
ready.


In [41]:

# ---------------------------------------------------------------------------
# CELL: benchmark the SOURCE data — validates the check pipeline AND gives the
# intrinsic floors a prediction must match (per sample, vs its own source).
#   field = DEVIATION u (u_tot = u + y, laminar U(y)=y).  Re=400, Re_tau~34.
# ---------------------------------------------------------------------------
import time

# stored wall/tau_mean per realization (Chebyshev-exact, total shear)
def get_tau_mean(real, t_slice):
    with h5py.File(H5, 'r') as f:
        a = np.array(f[real]['wall']['tau_mean_a'])[t_slice]
        b = np.array(f[real]['wall']['tau_mean_b'])[t_slice]
    return a, b

tau_lists_a, tau_lists_b = [], []
for r in REALS:
    a, b = get_tau_mean(r, T_SUB)
    tau_lists_a.append(a); tau_lists_b.append(b)
TAU_A = np.concatenate(tau_lists_a)   # [T_total]
TAU_B = np.concatenate(tau_lists_b)

t0 = time.time()
src_checks = run_all_checks(SRC, NU, tau_mean_a=TAU_A, tau_mean_b=TAU_B)
print(f"checks done in {time.time()-t0:.1f}s")

trackio.init(project="case1", config={"dataset": "small-B_SOURCE (floor reference)",
                                      "nu": NU, "n_t": SRC.shape[0], "grid": f"{Nx}x{Ny}x{Nz}"})
trackio.log({**flatten(src_checks, "src_"), "step": 0}, step=0)
trackio.finish()

for k, v in src_checks.items():
    print(k, " ", v)
print("""
FLOORS (a good prediction should match these, per sample, vs its own source):
  divergence  : DNS incompressibility residual ~0.2 mean|div| (interior ~93% of u_rms);
                2nd-order wall one-sided stencils add ~6%.
  wall_bc     : DNS enforces u_dev|_wall = 0 exactly (all 3 components).
  re_tau      : ~34 (stored wall/tau_mean, Chebyshev-exact, total shear).
  Q           : no stored baseline; compare pred-vs-source instead.
""")


checks done in 1.0s
* Created new run: happy-ocean-6
* Run finished. Uploading logs to Trackio (please wait...)
divergence   {'mean|div|': 0.19645163416862488, 'mean(div)': 2.1055898691280106e-10, 'max|div|': 1.7381953001022339}
wall_bc   {'wall_a |u| mean': 6.846080268711838e-16, 'wall_b |u| mean': 8.241285940253928e-16, 'wall_a u_y mean': 5.48859581981229e-17, 'wall_b u_y mean': 5.431540540753077e-17}
Q   {'Q mean': -0.040310174226760864, 'Q rms': 0.08286504447460175}
re_tau   {'u_tau': 0.08552372455596924, 'Re_tau': 34.209489822387695}
stationarity   {'d<field>/dt rms': 0.1270521730184555, 'rel. d<field>/dt': 0.7351176738739014}

FLOORS (a good prediction should match these, per sample, vs its own source):
  divergence  : DNS incompressibility residual ~0.2 mean|div| (interior ~93% of u_rms);
                2nd-order wall one-sided stencils add ~6%.
  wall_bc     : DNS enforces u_dev|_wall = 0 exactly (all 3 components).
  re_tau      : ~34 (stored wall/tau_mean, Chebyshev-exact, t

In [19]:

# ---------------------------------------------------------------------------
# CELL: s3fs helpers — save / load / list / delete prediction datasets
#   Layout:  simulations/couette_ds/<name>/
#       u.zarr            [T,3,X,Y,Z]   f32  (the predicted field)
#       source_zarr.npy   per-sample source-index mapping  [T]
# ---------------------------------------------------------------------------
import zarr as zr

def save_prediction(name, u, src_idx, grid=None):
    """u: [T,3,X,Y,Z] np f32 ; src_idx: [T] int ; grid: dict (metadata).
    `src_idx[i]` indexes the flattened source list (realizations in order,
    each sliced by t_sub) — used to compare each prediction against ITS source."""
    root = f"{PRED_ROOT}/{name}"
    m = s3fs.S3Map(root=f"{root}/u.zarr", s3=fs)
    z = zr.open_array(m, mode="w", shape=u.shape, dtype="float32",
                      chunks=(1, 3, *u.shape[2:]), compressor=zr.Blosc(cname="lz4", clevel=5))
    for i in range(u.shape[0]):
        z[i] = u[i]
    z.attrs.update({"grid": grid or {}, "name": name})
    with fs.open(f"{root}/src_idx.npy", "wb") as fh:
        fh.write(np.ascontiguousarray(np.asarray(src_idx, dtype=np.int64)).tobytes())
    return f"{root}/u.zarr"

def load_prediction(name):
    m = s3fs.S3Map(root=f"{PRED_ROOT}/{name}/u.zarr", s3=fs)
    z = zr.open(store=m, mode="r")
    src_idx = np.frombuffer(fs.cat(f"{PRED_ROOT}/{name}/src_idx.npy"), dtype=np.int64)
    return np.asarray(z[:], dtype=np.float32), src_idx, dict(z.attrs)

def list_predictions():
    out = []
    for p in fs.find(f"{PRED_ROOT}/"):
        if p["name"] == "u.zarr":
            out.append(p["name"].split("/")[-2])
    return sorted(set(out))

def delete_prediction(name):
    fs.rm(f"{PRED_ROOT}/{name}", recursive=True)
    print("deleted", f"{PRED_ROOT}/{name}")

print("existing prediction datasets:", list_predictions())


existing prediction datasets: []


In [ ]:

# ---------------------------------------------------------------------------
# CELL: benchmark a PREDICTION dataset vs ITS OWN source (per sample).
#   NAME = prediction name saved by save_prediction()  e.g. "m4_base"
#   Metrics:
#     pred_*        : run_all_checks on the whole prediction
#     src_*         : run_all_checks on the matched source samples (baseline)
#     vs_src_mse_*  : per-component + total RMSE of (pred - source)
#     div_err       : mean|div(pred) - div(source)|   (vs its own source)
#     wall_err      : mean|u| at both walls of (pred - source)
# ---------------------------------------------------------------------------
import time, trackio

NAME = "m4_base"            # <-- set to the saved prediction name
pred, src_idx, attrs = load_prediction(NAME)
print("prediction:", pred.shape, pred.dtype, "n_src_idx:", len(src_idx),
      "realizations covered:", sorted(set(REALS[i//5] for i in np.unique(src_idx))))

# matched source (per sample)
src_matched = SRC[src_idx]

t0 = time.time()
pred_checks = run_all_checks(pred, NU)
src_matched_checks = run_all_checks(src_matched, NU)
dt = time.time()-t0

# per-sample residuals vs own source
res = pred - src_matched                                   # [T,3,X,Y,Z]
per_comp = [np.sqrt((res[:, c]**2).mean()) for c in range(3)]
total_rmse = np.sqrt((res**2).mean())

# divergence error vs own source
def div_of(field):
    g = grad(field)
    return g[:, 0, 0] + g[:, 1, 1] + g[:, 2, 2]
div_err = np.abs(div_of(pred) - div_of(src_matched)).mean()

# wall residual vs own source
wall_err = np.mean([np.abs(res[:, :, 0, :]).mean(), np.abs(res[:, :, -1, :]).mean()])

trackio.init(project="case1", config={"dataset": f"small-B_PRED {NAME}", "nu": NU,
                                      "n_t": pred.shape[0], "grid": f"{Nx}x{Ny}x{Nz}"})
trackio.log({
    **flatten(pred_checks, "pred_"), **flatten(src_matched_checks, "src_"),
    "vs_src_rmse_total":  total_rmse,
    "vs_src_rmse_u":      per_comp[0],
    "vs_src_rmse_v":      per_comp[1],
    "vs_src_rmse_w":      per_comp[2],
    "vs_src_div_err":     div_err,
    "vs_src_wall_err":    wall_err,
    "step": 0,
}, step=0)
trackio.finish()

print(f"\npred {NAME}: total RMSE vs own source = {total_rmse:.4f}  (u_rms={np.sqrt((SRC**2).mean()):.4f})")
print(f"  per-comp: u={per_comp[0]:.4f} v={per_comp[1]:.4f} w={per_comp[2]:.4f}")
print(f"  div(pred) mean| | = {pred_checks['divergence']['mean|div|']:.4f}   "
      f"div(src) mean| | = {src_matched_checks['divergence']['mean|div|']:.4f}   "
      f"|div(pred)-div(src)| = {div_err:.4f}")
print(f"  wall|u| pred = {pred_checks['wall_bc']['wall_a |u| mean']:.4f}/{pred_checks['wall_bc']['wall_b |u| mean']:.4f}   "
      f"wall|u| src = {src_matched_checks['wall_bc']['wall_a |u| mean']:.4f}/{src_matched_checks['wall_bc']['wall_b |u| mean']:.4f}")
print(f"  Re_tau pred = {pred_checks['re_tau']['Re_tau']:.1f}   src = {src_matched_checks['re_tau']['Re_tau']:.1f}")
print(f"  ({dt:.1f}s)")
